# 🧠 Fine-Tuning DistilRoBERTa for Emotion Classification

**Project:** Mental Health Agentic AI Platform  
**Author:** Jiya Yadav (@JIYA-YDV)  
**Goal:** Improve baseline macro F1 (0.87) by fine-tuning on `dair-ai/emotion` train split.

---

## 📋 Execution Order

1. **Setup:** Install dependencies + verify GPU
2. **Auth:** Log into HuggingFace Hub
3. **Data:** Load + preprocess `dair-ai/emotion`
4. **Model:** Load base DistilRoBERTa with classification head
5. **Train:** Fine-tune for 3 epochs (~25-40 min on T4 GPU)
6. **Evaluate:** Sanity-check on validation set
7. **Push:** Upload final model to your HF account
8. **Download:** Get model files for local integration


## 1️⃣ Install Dependencies

In [1]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 evaluate==0.4.3 scikit-learn==1.5.1
print('✅ Dependencies installed')

✅ Dependencies installed



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import transformers
import datasets as hf_datasets

print(f'Torch version: {torch.__version__}')
print(f'Transformers version: {transformers.__version__}')
print(f'Datasets version: {hf_datasets.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ NO GPU DETECTED — Go to Runtime → Change runtime type → T4 GPU')

C:\Users\yoga\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.12.1+cpu
Transformers version: 4.44.2
Datasets version: 2.21.0
CUDA available: False
⚠️ NO GPU DETECTED — Go to Runtime → Change runtime type → T4 GPU


## 2️⃣ Authenticate with HuggingFace Hub

When prompted, paste your token (starts with `hf_`). Get it from https://huggingface.co/settings/tokens (use a **Write** scope token).

In [ ]:
from huggingface_hub import login
login()  # Will prompt for token

## 3️⃣ Load & Preprocess Dataset

**Dataset:** `dair-ai/emotion` — 16K Twitter posts labeled with 6 emotions:
- 0: sadness
- 1: joy
- 2: love
- 3: anger
- 4: fear
- 5: surprise

In [ ]:
from datasets import load_dataset
from collections import Counter

ds = load_dataset('dair-ai/emotion')
print('Dataset structure:')
print(ds)
print()
print('Label distribution (train):')
label_names = ds['train'].features['label'].names
train_counts = Counter(ds['train']['label'])
for idx, name in enumerate(label_names):
    print(f'  {name:10s} : {train_counts[idx]:5d}')

In [ ]:
# Show some examples
for i in range(3):
    sample = ds['train'][i]
    label_name = label_names[sample['label']]
    print(f'[{label_name:10s}] {sample["text"]}')

## 4️⃣ Load Tokenizer & Model

We start from `j-hartmann/emotion-english-distilroberta-base` which already has 7-class emotion knowledge — fine-tuning will adapt its weights to the 6-class dair-ai dataset distribution.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_MODEL = 'j-hartmann/emotion-english-distilroberta-base'
NUM_LABELS = 6  # dair-ai/emotion has 6 classes

# Label mappings for the new head
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # because we're changing the classification head
)

print(f'✅ Loaded {BASE_MODEL}')
print(f'Model parameters: {model.num_parameters():,}')
print(f'Label mapping: {id2label}')

## 5️⃣ Tokenize Dataset

In [ ]:
MAX_LENGTH = 128  # Most tweets are under 50 tokens; 128 is plenty

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=False,  # dynamic padding via DataCollator (more efficient)
        max_length=MAX_LENGTH,
    )

tokenized_ds = ds.map(tokenize_function, batched=True, remove_columns=['text'])
print('✅ Tokenized')
print(tokenized_ds)

## 6️⃣ Configure Training

**Hyperparameters chosen for speed + quality balance:**
- 3 epochs (sufficient for 16K samples; more risks overfitting)
- Learning rate 2e-5 (standard for transformer fine-tuning)
- Batch size 32 (fits T4 GPU memory comfortably)
- Linear warmup over first 10% of steps
- Weight decay 0.01 for regularization

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average='macro')
    weighted_f1 = f1_score(labels, predictions, average='weighted')
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
    }

OUTPUT_DIR = './finetuned-emotion'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,  # only keep best 2 checkpoints (saves disk)
    logging_dir='./logs',
    logging_steps=50,
    report_to='none',  # disable wandb / tensorboard upload
    seed=42,
    fp16=True,  # mixed precision — 2x speedup on T4 GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('✅ Trainer configured')
print(f'Total training steps: {len(tokenized_ds["train"]) // 32 * 3}')

## 7️⃣ Train! 🚀

**Expected duration:** 25-40 minutes on T4 GPU.

You'll see loss + macro_f1 logged every 50 steps and after each epoch.

In [ ]:
train_result = trainer.train()

print()
print('=' * 60)
print('✅ TRAINING COMPLETE')
print('=' * 60)
print(f'Total training time: {train_result.metrics["train_runtime"]:.1f} seconds')
print(f'Final training loss: {train_result.metrics["train_loss"]:.4f}')

## 8️⃣ Evaluate on Test Set (Sanity Check)

In [ ]:
test_results = trainer.evaluate(tokenized_ds['test'])

print()
print('=' * 60)
print('📊 TEST SET RESULTS')
print('=' * 60)
for key, value in test_results.items():
    if isinstance(value, float):
        print(f'  {key:.<40s} {value:.4f}')
print()
print('🎯 Compare to your baseline (j-hartmann/emotion): macro F1 = 0.8693')

## 9️⃣ Detailed Per-Class Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get predictions on test set
predictions_output = trainer.predict(tokenized_ds['test'])
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

print('Per-class classification report:')
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title('Fine-Tuned Model — Confusion Matrix')
plt.ylabel('Ground Truth')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix_finetuned.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔟 Push Model to HuggingFace Hub

This uploads your fine-tuned model to `https://huggingface.co/YDVJIYA/distilroberta-emotion-finetuned`

In [ ]:
HF_USERNAME = 'YDVJIYA'  # Your HF username
REPO_NAME = 'distilroberta-emotion-finetuned'
HUB_MODEL_ID = f'{HF_USERNAME}/{REPO_NAME}'

# Build a model card
model_card_content = f'''---
license: mit
tags:
- emotion-classification
- distilroberta
- fine-tuned
- mental-health
datasets:
- dair-ai/emotion
metrics:
- f1
- accuracy
base_model: j-hartmann/emotion-english-distilroberta-base
language:
- en
---

# DistilRoBERTa Fine-Tuned for Emotion Classification

Fine-tuned from `j-hartmann/emotion-english-distilroberta-base` on the `dair-ai/emotion` dataset.

Part of the [Mental Health Agentic AI Platform](https://github.com/JIYA-YDV/Mental-Health-Agentic-AI-Platform).

## Performance

| Metric | Baseline | Fine-Tuned |
|--------|----------|------------|
| Macro F1 | 0.869 | **{test_results["eval_macro_f1"]:.3f}** |
| Accuracy | 0.914 | **{test_results["eval_accuracy"]:.3f}** |

## Labels

- 0: sadness
- 1: joy
- 2: love
- 3: anger
- 4: fear
- 5: surprise

## Usage

```python
from transformers import pipeline
classifier = pipeline("text-classification", model="{HUB_MODEL_ID}")
result = classifier("I feel hopeless and tired today")
print(result)
```

## Training

- Epochs: 3
- Batch size: 32
- Learning rate: 2e-5
- Warmup ratio: 0.1
- Weight decay: 0.01
- Mixed precision (fp16)
- Seed: 42

## Author

Trained by [Jiya Yadav](https://github.com/JIYA-YDV) as part of an end-to-end agentic AI platform.
'''

# Save model card
with open(f'{OUTPUT_DIR}/README.md', 'w', encoding='utf-8') as f:
    f.write(model_card_content)

# Push everything to Hub
trainer.push_to_hub(
    repo_id=HUB_MODEL_ID,
    commit_message='Initial fine-tuned model upload',
)

print()
print(f'✅ Model published at: https://huggingface.co/{HUB_MODEL_ID}')

## 1️⃣1️⃣ Bundle Model for Local Download

Creates a ZIP file you can download from Colab and use locally.

**Why both Hub AND local copy?**
- Hub copy = shareable, versioned, public model card
- Local copy = fast offline inference, no internet dependency in your platform

In [ ]:
import shutil
import os

# Save just the model + tokenizer (not the checkpoints, logs, etc.)
FINAL_MODEL_DIR = './model_to_download'
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Show what's in it
print('Files to download:')
for f in os.listdir(FINAL_MODEL_DIR):
    size_mb = os.path.getsize(os.path.join(FINAL_MODEL_DIR, f)) / 1e6
    print(f'  {f:30s}  {size_mb:.1f} MB')

# Create a ZIP
shutil.make_archive('finetuned_emotion_model', 'zip', FINAL_MODEL_DIR)
zip_size_mb = os.path.getsize('finetuned_emotion_model.zip') / 1e6
print(f'\n✅ Created finetuned_emotion_model.zip ({zip_size_mb:.1f} MB)')
print()
print('📥 To download:')
print('   Click the folder icon on the left sidebar → right-click on')
print('   finetuned_emotion_model.zip → Download')

## 1️⃣2️⃣ Summary & Next Steps

### ✅ What You Accomplished

1. Fine-tuned DistilRoBERTa on 16K labeled emotion samples
2. Evaluated on held-out test set
3. Generated confusion matrix
4. Published model to HuggingFace Hub
5. Bundled local copy for download

### 📥 Next Steps (Back in Your Local Repo)

1. Download `finetuned_emotion_model.zip` from Colab
2. Extract to `models/finetuned-emotion/` in your repo
3. Update `backend/config/settings.py`:
   ```python
   EMOTION_MODEL = "./models/finetuned-emotion"  # or YDVJIYA/distilroberta-emotion-finetuned
   ```
4. Re-run your evaluation harness:
   ```bash
   python -m evaluation.benchmark --samples 1000
   ```
5. Compare the new `docs/EVALUATION.md` against the old one

### 🎯 Resume Bullet (Update with Real Numbers)

> Fine-tuned DistilRoBERTa on 16K labeled emotion samples using AdamW + linear warmup,
> achieving macro F1 of **[YOUR_SCORE]** vs 0.87 baseline. Published model to
> [HuggingFace Hub](https://huggingface.co/YDVJIYA/distilroberta-emotion-finetuned)
> with reproducible training pipeline on Google Colab GPU.